# Taller — Deep Learning: ¿quién es esta persona?
**Machine Learning · Universidad Libre · Gustavo Santos** · Trabajo en parejas · 30 min + 5 min de socialización

Dataset: **Olivetti Faces** (400 fotos 64×64, 40 personas × 10 fotos).
Van a comparar **tres modelos del curso sobre el mismo problema de imágenes, con la misma partición**:

| # | Modelo | Conexión con la clase |
|---|---|---|
| 1 | Red densa (MLPClassifier) | La red que entrenamos con `load_digits` |
| 2 | SVM con kernel RBF + PCA | Parte A de hoy |
| 3 | Red convolucional (CNN, Keras) | La demo de rostros |

**Entregable:** este notebook ejecutado + la tabla comparativa (punto 4) + el párrafo de reflexión (punto 5).
Las celdas marcadas con `# TODO` son las que ustedes deben completar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

tf.keras.utils.set_random_seed(42)   # reproducibilidad (numpy + python + TF)
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 0. Datos y partición (ya resuelto — no modificar)
Todos los modelos usan exactamente la misma partición: la comparación es justa.

In [ ]:
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
images, y = faces.images, faces.target
print("Shape imágenes:", images.shape)            # (400, 64, 64)
print("Personas (clases):", len(np.unique(y)))    # 40
print("Imágenes por persona:", np.bincount(y)[:5], "...")   # 10 c/u
print("Rango de píxeles:", images.min(), "-", images.max())  # ya vienen en [0, 1]

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(images[i], cmap="gray")
    plt.title(f"Persona {y[i]}")
    plt.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Tensor de rango 4 para la CNN: (n_imágenes, alto, ancho, canales)
X = images[..., np.newaxis]          # (400, 64, 64, 1)

# 1) Separamos PRUEBA (20%) — no se toca hasta el final
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42)

# 2) Del resto separamos VALIDACIÓN (20%) para monitorear el entrenamiento
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42)

print("Entrenamiento:", X_train.shape, "| Validación:", X_val.shape, "| Prueba:", X_test.shape)
# stratify=y garantiza 2 fotos de cada una de las 40 personas en prueba

# Versión "aplanada" para los modelos de sklearn: cada foto = vector de 64×64 = 4096 píxeles
Xtr_flat  = X_train.reshape(len(X_train), -1)
Xval_flat = X_val.reshape(len(X_val), -1)
Xte_flat  = X_test.reshape(len(X_test), -1)
print("Aplanado:", Xtr_flat.shape)

In [ ]:
def graficar_historia(history, titulo=""):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(history.history["accuracy"], label="entrenamiento")
    ax[0].plot(history.history["val_accuracy"], label="validación")
    ax[0].set_title(f"Accuracy {titulo}"); ax[0].set_xlabel("Época"); ax[0].legend()
    ax[1].plot(history.history["loss"], label="entrenamiento")
    ax[1].plot(history.history["val_loss"], label="validación")
    ax[1].set_title(f"Pérdida (cross-entropy) {titulo}"); ax[1].set_xlabel("Época"); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
resultados = {}   # aquí guardaremos la accuracy en PRUEBA de cada modelo

## 1. Línea base: red densa (MLP) — ~5 min
Entrenen un `MLPClassifier` sobre las fotos aplanadas, igual que con `load_digits`.
- Escalen con `StandardScaler` (ajustado **solo** con entrenamiento).
- Prueben una arquitectura, por ejemplo `hidden_layer_sizes=(256,)`, `max_iter=500`, `random_state=42`.
- Reporten accuracy en prueba y guárdenla en `resultados["MLP"]`.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

# TODO: escalar
scaler = StandardScaler()
Xtr_s = ...
Xte_s = ...

# TODO: definir y entrenar el MLP
mlp = MLPClassifier(...)
...

# TODO: accuracy en prueba
acc_mlp = ...
resultados["MLP"] = acc_mlp
print(f"MLP — accuracy en prueba: {acc_mlp:.3f}")

## 2. SVM + PCA ("eigenfaces") — ~8 min
4096 variables para 256 fotos es mucho. Primero reducimos dimensión con **PCA** (por ejemplo a 100 componentes) y luego aplicamos **SVM con kernel RBF**.
- Armen un `Pipeline` con `StandardScaler` → `PCA(n_components=100, whiten=True, random_state=42)` → `SVC(kernel="rbf")`.
- Hagan un `GridSearchCV` (cv=3) sobre al menos 2 valores de `C` y 2 de `gamma`.
- Reporten el mejor par (C, gamma) y la accuracy en prueba → `resultados["SVM + PCA"]`.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=100, whiten=True, random_state=42)),
    ("svm", SVC(kernel="rbf")),
])

# TODO: grilla de hiperparámetros (los nombres llevan el prefijo del paso: "svm__C", "svm__gamma")
param_grid = {...}

# TODO: GridSearchCV y entrenamiento (usen X_temp aplanado = entrenamiento + validación)
Xtemp_flat = X_temp.reshape(len(X_temp), -1)
grid = ...

# TODO: mejores parámetros y accuracy en prueba
print("Mejores parámetros:", ...)
acc_svm = ...
resultados["SVM + PCA"] = acc_svm
print(f"SVM + PCA — accuracy en prueba: {acc_svm:.3f}")

## 3. CNN: mejorar la red de la demo — ~12 min
Partan de la CNN de la demo y hagan **al menos UNO** de estos cambios. Comparen las curvas contra las de la demo.
- **(a)** *Data augmentation*: agregar al inicio `keras.layers.RandomFlip("horizontal")`, `RandomRotation(0.05)`, `RandomZoom(0.1)`.
- **(b)** Cambiar la arquitectura: un tercer bloque `Conv2D + MaxPooling2D`, o más/menos filtros.
- **(c)** Cambiar la regularización: otro valor de `Dropout`, o quitarlo y ver qué pasa.

Usen `EarlyStopping` y evalúen **una sola vez** en prueba → `resultados["CNN"]`.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(64, 64, 1)),
    # TODO (a): capas de data augmentation aquí (opcional)

    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    # TODO (b): bloque convolucional adicional (opcional)

    keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.Dropout(0.5),          # TODO (c): probar otro valor
    keras.layers.Dense(40, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

early = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, start_from_epoch=15, restore_best_weights=True)
# TODO: entrenar con validation_data=(X_val, y_val), epochs=100, callbacks=[early]
history = ...

graficar_historia(history, "— mi CNN")

In [ ]:
# TODO: evaluar en prueba
acc_cnn = ...
resultados["CNN"] = acc_cnn
print(f"CNN — accuracy en prueba: {acc_cnn:.3f}")
print("Cambio(s) realizado(s): ...")   # TODO: describir qué cambiaron

## 4. Tabla comparativa

In [ ]:
import pandas as pd
tabla = pd.DataFrame({"Accuracy (prueba)": resultados}).sort_values("Accuracy (prueba)", ascending=False)
tabla["Errores (de 80)"] = ((1 - tabla["Accuracy (prueba)"]) * len(y_test)).round().astype(int)
tabla

## 5. Reflexión (un párrafo por pareja)
1. ¿Cuál modelo ganó en **este** problema y por qué creen que pasó? (Piensen en qué ignora una red densa al aplanar la imagen.)
2. Comparen con lo visto en crédito: allí la regresión logística le ganó a SVM. ¿Qué cambia entre un problema tabular y uno de imágenes?
3. ¿En qué escenario de su propio interés profesional usarían SVM en lugar de una red neuronal, y por qué?

**Respuesta:**

*(escriban aquí)*